## What This Demonstrates

```text
The demo demonstrates my ability to:

- identify repetitive operational bottlenecks
- model workflows as structured relational data
- combine extracts from different systems into a unified logical flow
- automate repetitive administrative tasks
- reduce manual work and assignment errors
- create deterministic and reproducible outputs

The goal was not to create a complex system, but to simplify and standardize a repetitive process in a safe and scalable way.
```

## Requirements

```text
Python 3.x
Jupyter Notebook
pandas
```

```text
Jupyter Notebook is used to run the demo interactively as a notebook.

pandas is used for loading and exporting CSV files and for working smoothly with the SQLite3 in-memory database.

Any modern Python 3.x version should work well.
```

# Files

## access_rules.csv

```text
Defines ward access and ward rotation rules.

employed_at,access_unit,access_type
A1,A1,N
A1,A3,R
A2,A1,R
...

Description:

employed_at  = user's home ward
access_unit  = ward the user can access
access_type  = N (normal/home access) or R (rotation/remote access)
```

## equipment.csv

```text
Defines which instrument classes are available at each ward.

unit_id,equipment_class
A1,ALPHA
A1,BETA
A2,BETA
...

Description:

unit_id         = ward identifier
equipment_class = instrument or analyzer class available at the ward
```

## users.csv
```text
Defines the users to be imported.

user_id,name,employed_at
U1,Alice,A1
U2,Bob,A2
...

Description:

user_id     = unique user identifier
name        = user name
employed_at = user's home ward
```

## user_import_export.csv

```text
Created import file that is read by the Point of Care Laboratory Information System import script.

user_id,user_name,employed_at,access_unit,access_type,equipment_class
U1,Alice,A1,A1,N,ALPHA
...


## demo.html
```text
This is a simple in-browser HTML file that mimics the manual process of assigning user access.

The demo illustrates how repetitive and error-prone manual assignment can become when every access row must be entered individually.
```

# Code

In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(":memory:")

pd.read_csv("users.csv").to_sql("users", conn, index=False, if_exists="replace")
pd.read_csv("access_rules.csv").to_sql("access_rules", conn, index=False, if_exists="replace")
pd.read_csv("equipment.csv").to_sql("equipment", conn, index=False, if_exists="replace")

query = """
SELECT
    u.user_id,
    u.name AS user_name,
    u.employed_at,
    ar.access_unit,
    ar.access_type,
    e.equipment_class

FROM users u

JOIN access_rules ar
    ON ar.employed_at = u.employed_at

JOIN equipment e
    ON e.unit_id = ar.access_unit

ORDER BY
    u.user_id,
    ar.access_unit,
    e.equipment_class
"""

rows = pd.read_sql_query(query, conn)

rows.to_csv(
    "user_import_export.csv",
    index=False,
    sep=";"
)

print(rows)

   user_id user_name employed_at access_unit access_type equipment_class
0       U1     Alice          A1          A1           N           ALPHA
1       U1     Alice          A1          A1           N            BETA
2       U1     Alice          A1          A1           N           GAMMA
3       U1     Alice          A1          A3           R           DELTA
4       U2       Bob          A2          A1           R           ALPHA
5       U2       Bob          A2          A1           R            BETA
6       U2       Bob          A2          A1           R           GAMMA
7       U2       Bob          A2          A2           N            BETA
8       U2       Bob          A2          A2           N           GAMMA
9       U2       Bob          A2          A4           R           ALPHA
10      U3   Charlie          A3          A1           R           ALPHA
11      U3   Charlie          A3          A1           R            BETA
12      U3   Charlie          A3          A1       

# User access is determined by:

```text
user home ward
→ ward access / rotation rules
→ target ward
→ instrument classes available at that ward

The system administrator defines wards and instruments in the point-of-care system, while laboratory staff assign users to wards. Since ward access, staff rotation, and instrument placement are known structured information, they can be standardized into CSV files and combined using SQL.
```

# Purpose

```text
The purpose of the demo is to show how repetitive administrative work can be reduced by moving logic from manual UI interaction into structured tables and deterministic joins.

The same principle can be applied whenever known relationships between users, organizational units, roles, equipment, or products need to be transformed into a vendor-specific import format.
```

# Key Concept

### Manual process:

```text
Add user manually
→ assign ward access manually
→ assign instrument access manually
→ repeat for every user
```

### Automated process:

```text
Add user to CSV
→ run Python script
→ generate import file
→ vendor import script assigns access
```

# Core Idea
```text
- users
- access_rules
- equipment_classes_by_ward

The import file is generated by joining these tables.
```
### This makes the process:
```text
- reproducible
- transparent
- easier to validate
- less dependent on manual work
- less prone to assignment errors
```
